# Feature Exploration — Temperature Forecasting

Before training the LSTM, it's worth understanding what signal is actually
sitting in the dataset `real_data/fetch_data.py` produced. This notebook
walks through exploring `real_data/berlin_data.csv` (or whichever station
you fetched): looking at correlations, variance, and other summary
statistics for each feature relative to the **target** you're trying to
forecast, `temperature_2m`.

Each section below has a short explanation and a task. The code cells are
left empty on purpose — write your own code there to complete the task
before moving to the next section.

## 1. Load the data

`fetch_data.py` saved the aligned truth + NWP-forecast dataset to
`real_data/<station>_data.csv`, indexed by timestamp.

**Task:** Load the CSV with pandas. Make sure the date column becomes a
proper `DatetimeIndex` (not just a string column) — you'll need that for
the time-based work later in this notebook.

## 2. Get familiar with the data

Before any modeling, always check the basics — shape, column types,
missing values, and the date range actually covered. These often reveal
fetch bugs or gaps you'd otherwise miss.

**Task:**
- Print the shape of the DataFrame and its column dtypes.
- Count missing values per column.
- Print the first and last timestamp to confirm the covered date range.

## 3. Visualize the target over time

Plotting the raw series is the fastest way to spot obvious patterns —
daily cycles, seasonal trend, missing chunks, or outliers — before diving
into summary numbers.

**Task:**
- Plot `temperature_2m` against the date index for the full dataset.
- Then plot just a single week of it, so the daily cycle is clearly
  visible.

## 4. Distribution of each feature

A histogram shows you the spread, skew, and any suspicious outliers in a
feature — things a single mean/std number can hide.

**Task:**
- Plot a histogram for every numeric column (`DataFrame.hist()` does this
  for all columns at once).
- Note anything that looks off: an unexpected spike, a suspiciously flat
  distribution, or values that seem out of physical range.

## 5. Correlation coefficients with the target

The Pearson correlation coefficient measures the strength of the
*linear* relationship between two variables, from -1 (perfect negative)
to +1 (perfect positive), with 0 meaning no linear relationship. Keep in
mind: two variables can have a strong *non-linear* relationship and still
show a weak Pearson correlation, so this isn't the whole picture — just a
useful first pass.

**Task:**
- Compute the Pearson correlation of every feature with `temperature_2m`
  (`DataFrame.corr()`), and sort the results by absolute value.
- Which feature correlates most strongly with the target? Does that match
  your intuition?
- Plot the full correlation matrix as a heatmap so you can see how
  features relate to *each other*, not just to the target.

## 6. Variance and spread of each feature

Variance measures how spread out a feature's values are around its mean.
A feature with (near) zero variance carries almost no information — it
barely changes, so it can't help a model distinguish one moment from
another, no matter how "correlated" it superficially looks.

Because these features live on very different scales (temperature in °C
vs. pressure in hPa), raw variance isn't directly comparable across
columns. The **coefficient of variation** (`std / mean`) is a scale-free
way to compare spread across variables with different units.

**Task:**
- Compute the variance and standard deviation of each numeric column.
- Compute the coefficient of variation for each column and compare it
  across variables.
- Are there any near-constant columns that might not be useful for the
  model?

## 7. Autocorrelation of the target

Since this is a time series, how strongly a value relates to its *own*
past values (autocorrelation) matters a lot — it directly informs how
large a lookback window the LSTM needs to see useful history. The
autocorrelation at lag *k* is the correlation between the series and a
copy of itself shifted by *k* steps (`Series.autocorr(lag=k)`).

**Task:**
- Compute the autocorrelation of `temperature_2m` at lags of 1, 6, 12,
  24, 48, and 168 hours (1 hour up to 1 week).
- Plot autocorrelation vs. lag. At what lag does it drop off? What does
  that suggest about how many past hours the LSTM's input window should
  cover?

*(Optional stretch: `statsmodels.graphics.tsaplots.plot_acf` gives a
nicer ACF plot with confidence intervals built in.)*

## 8. How good is the NWP baseline already?

The `temperature_2m_previous_day{N}` columns aren't just another
feature — they're the archived operational NWP forecast made N days
before each timestamp. Comparing them against the true `temperature_2m`
tells you how strong a "physics baseline" the LSTM will eventually be
measured against.

**Task:**
- For each `temperature_2m_previous_day{N}` column, compute its
  correlation with `temperature_2m`, and its RMSE against
  `temperature_2m`.
- How do correlation and RMSE change as the lead time N grows? Does that
  match your expectations about forecast skill decaying with lead time?

## 9. Summary

Write a short summary of what you found. There's no single right
answer here — the point is to reason from what you actually saw above.

**Your answers:**

- Which features look most useful for forecasting temperature?

  *(your answer here)*

- Are there any features you'd drop, or engineer differently (e.g.
  combine, transform)?

  *(your answer here)*

- Based on the autocorrelation, how far back should the LSTM's input
  window look?

  *(your answer here)*

- How big a gap does the LSTM need to close relative to the NWP
  baseline?

  *(your answer here)*